In [ ]:
import os
import sys
import openai
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from pathlib import Path

In [ ]:
from google.oauth2 import service_account
from googleapiclient.discovery import build

# Auth
SERVICE_ACCOUNT_FILE = '/Users/daianeklein/Documents/DS/job-applications-tool/h.json'
SCOPES = ["https://www.googleapis.com/auth/documents.readonly"]
creds = service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=SCOPES)

# Docs API Service
service = build('docs', 'v1', credentials=creds)

DOCUMENT_ID = '1pXc4nsuFd5RfQFWimmKaLMxucWiV7WLZDCtP18wbCTE'

def fetch_cv_text():
    """Fetches the CV document content and extracts text."""
    doc = service.documents().get(documentId=DOCUMENT_ID).execute()

    def extract_text(document):
        text = []
        for element in document.get("body", {}).get("content", []):
            if "paragraph" in element:
                for paragraph_element in element["paragraph"]["elements"]:
                    if "textRun" in paragraph_element:
                        text.append(paragraph_element["textRun"]["content"])
        return "".join(text)

    return extract_text(doc)


if __name__ == '__main__':
    document_text = fetch_cv_text()
    print("\nDocument Content:\n", document_text)



Document Content:
 Daiane Klein
Senior Data Analyst | Analytics Engineer

São Paulo, Brazil    |   +55 11 962192070    |    Linkedin    |     Github
PROFILE SUMMARY
Data Analyst with 7+ years of experience in Data Analysis, including business and customer insights, in different industries. Proficient in Python, SQL, and Dashboard development. Strong understanding of Machine Learning, statistics, and Large Language Models (LLMs) as well as business impact and results.

 	SKILLS
	Professional Skills: 	Data Analysis | Data Science | Data Visualization | Artificial Intelligence | LLMs | Data 
Engineering | Data Quality | ETL | Data Pipelines
	Stacks & Tools: 	Python | SQL | Excel | Cloud | Docker | Snowflake | dbt | GIT | Power BI | Sigma Computing |
N8N | AWS | Prefect
Languages: 		Portuguese (Native) | English (C1 Advanced)

 	WORK EXPERIENCE
	AI & Data Strategy Consultant, Stealth AI Startup  (Contract)					Jan 2025 - Present
Developed an LLM-powered pipeline to analyze unstructured da

In [10]:
load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

if not OPENAI_API_KEY:
    raise ValueError('OPENAI API KEY NOT FOUND')

# Initialize OpenAI Chat Model
llm = ChatOpenAI(model_name='gpt-4o', openai_api_key=OPENAI_API_KEY)

In [5]:
doc = service.documents().get(documentId=DOCUMENT_ID).execute()
doc

{'title': 'daiane-klein-resume-template',
 'body': {'content': [{'endIndex': 1,
    'sectionBreak': {'sectionStyle': {'columnSeparatorStyle': 'NONE',
      'contentDirection': 'LEFT_TO_RIGHT',
      'sectionType': 'CONTINUOUS'}}},
   {'startIndex': 1,
    'endIndex': 14,
    'paragraph': {'elements': [{'startIndex': 1,
       'endIndex': 14,
       'textRun': {'content': 'Daiane Klein\n',
        'textStyle': {'bold': True,
         'fontSize': {'magnitude': 22, 'unit': 'PT'},
         'weightedFontFamily': {'fontFamily': 'Lato', 'weight': 400}}}}],
     'paragraphStyle': {'namedStyleType': 'NORMAL_TEXT',
      'alignment': 'CENTER',
      'direction': 'LEFT_TO_RIGHT',
      'avoidWidowAndOrphan': False}}},
   {'startIndex': 14,
    'endIndex': 55,
    'paragraph': {'elements': [{'startIndex': 14,
       'endIndex': 55,
       'textRun': {'content': 'Senior Data Analyst | Analytics Engineer\n',
        'textStyle': {'weightedFontFamily': {'fontFamily': 'Lato',
          'weight': 300}}

In [6]:
for element in doc.get("body", {}).get("content", []):
    print(element)


{'endIndex': 1, 'sectionBreak': {'sectionStyle': {'columnSeparatorStyle': 'NONE', 'contentDirection': 'LEFT_TO_RIGHT', 'sectionType': 'CONTINUOUS'}}}
{'startIndex': 1, 'endIndex': 14, 'paragraph': {'elements': [{'startIndex': 1, 'endIndex': 14, 'textRun': {'content': 'Daiane Klein\n', 'textStyle': {'bold': True, 'fontSize': {'magnitude': 22, 'unit': 'PT'}, 'weightedFontFamily': {'fontFamily': 'Lato', 'weight': 400}}}}], 'paragraphStyle': {'namedStyleType': 'NORMAL_TEXT', 'alignment': 'CENTER', 'direction': 'LEFT_TO_RIGHT', 'avoidWidowAndOrphan': False}}}
{'startIndex': 14, 'endIndex': 55, 'paragraph': {'elements': [{'startIndex': 14, 'endIndex': 55, 'textRun': {'content': 'Senior Data Analyst | Analytics Engineer\n', 'textStyle': {'weightedFontFamily': {'fontFamily': 'Lato', 'weight': 300}}}}], 'paragraphStyle': {'namedStyleType': 'NORMAL_TEXT', 'alignment': 'CENTER', 'direction': 'LEFT_TO_RIGHT', 'avoidWidowAndOrphan': False}}}
{'startIndex': 55, 'endIndex': 56, 'paragraph': {'element

In [ ]:
def fetch_job_title():
    """Fetches the job title from the CV document in Google Docs."""
    doc = service.documents().get(documentId=DOCUMENT_ID).execute()
    
    content = doc.get("body", {}).get("content", [])
    
    job_title = None
    paragraph_count = 0  # Track which paragraph we're processing

    for element in content:
        if "paragraph" in element:
            paragraph_count += 1  # Increment for each paragraph
            
            # The second paragraph should be the job title
            if paragraph_count == 6:
                for paragraph_element in element["paragraph"]["elements"]:
                    if "textRun" in paragraph_element:
                        if paragraph_element["textRun"]["content"]:
                            job_title = paragraph_element["textRun"]["content"]
                            break
                        else:
                            continue
    return job_title

# Usage
job_title = fetch_job_title()
print("Found: ", job_title)


Found:  Data Analyst with 7+ years of experience in Data Analysis, including business and customer insights, in different industries. Proficient in Python, SQL, and Dashboard development. Strong understanding of Machine Learning, statistics, and Large Language Models (LLMs) as well as business impact and results.


In [11]:
update_job_title_p = '''
You are an AI assistant specializing in resume optimization.
Your task is to modify a given job title by updating it to align with the keywords provided.

## **Instructions:**  
1. **Preserve Truthfulness:** Do not invent or modify experiences, skills, or qualifications. Only adjust the job title if it accurately reflects the
candidate's role and responsibilities.  
2. **Ensure Industry Alignment:** The updated job title should match common industry standards for similar roles.  
3. **Maintain Formatting:** The output should maintain the original structure of the resume.  
4. **Only relevant roles** : You should choose up to 2 relevant roles, which are the most common roles and more aligned to the keywords provided.

## **Input:**  
- **The job title (as plain text/string)**  
- **Keywords with possibly job titles**  

## **Output Format:**  
Return a string containig the new job title.  

## **Example1**
Input: Senior Data Analyst | Analytics Engineer

Keywords:
Job Title Keywords: Senior Data Engineer, BI Developer
Hard Skills: Microsoft Power BI, Microsoft Azure, SQL, DAX, Azure Data Factory, Databricks, Interactive Analytics, Big Data, Cloud Environment, Tableau
Soft Skills: Problem-Solving, Collaboration, Communication, Consulting, Business Acumen
Industry-Specific Terms: KPIs, Semantic Models, Business Intelligence, Data Workloads, Workshops, Whiteboarding Sessions, Knowledge Transfer, IT Architecture

Output: Senior Data Engineer | BI Developer

## **Example2**
Input: Senior Data Analyst | Analytics Engineer

Keywords:
Job Title Keywords: Data Analyst, Analytics Specialist, IT Service Management Analyst
Hard Skills: Power BI, Tableau, Excel, SQL, Python, Data Visualization, Reporting, Power Automate, ITIL Framework, Market Intelligence, Azure
Soft Skills: Communication, Interpersonal Skills, Analytical Thinking, Problem-Solving, Attention to Detail, Proactivity, Time Management, Independent Working
Industry-Specific Terms: IT Service Management (ITSM), ITSM Enablement, Incident Management, Change Management, Problem Management, Automation, Global IT Policies

Output: Senior Data Analyst | Analytics Specialist

'''

In [12]:
def update_job_title(keywords:str, job_title:str) -> str:
    messages = [
        SystemMessage(content=update_job_title_p),
        HumanMessage(content=keywords),
        HumanMessage(content=job_title)
    ]

    response = llm.invoke(messages)
    return response.content.strip()

In [13]:
job_title = 'Job Title: Senior Data Analyst | Analytics Engineer'
keywords = '''Job Title Keywords: Senior Data Engineer, BI Developer
Hard Skills: Microsoft Power BI, Microsoft Azure, SQL, DAX, Azure Data Factory, Databricks, Interactive Analytics, Big Data, Cloud Environment, Tableau
Soft Skills: Problem-Solving, Collaboration, Communication, Consulting, Business Acumen
Industry-Specific Terms: KPIs, Semantic Models, Business Intelligence, Data Workloads, Workshops, Whiteboarding Sessions, Knowledge Transfer, IT Architecture
'''

update_job_title(keywords, job_title)

'Senior Data Engineer | BI Developer'

In [ ]:
def update_resume_new_title(new_title: str, existing_title: str):
    # Find the position of the job title
    start_index = None
    end_index = None

    for element in doc.get("body", {}).get("content", []):
        if "paragraph" in element:
            for paragraph_element in element["paragraph"]["elements"]:
                if "textRun" in paragraph_element:
                    text = paragraph_element["textRun"]["content"]
                    if existing_title in text:
                        start_index = paragraph_element["startIndex"]
                        end_index = paragraph_element["endIndex"]
                        break

    # If job title is found, update it
    if start_index and end_index:
        requests = [
            {
                "replaceAllText": {
                    "containsText": {
                        "text": existing_title,
                        "matchCase": True
                    },
                    "replaceText": new_title
                }
            }
        ]

        # Send update request
        service.documents().batchUpdate(documentId=UPDATED_DOC_ID, body={"requests": requests}).execute()

        print(f"Job title updated to: {new_title}")
    else:
        print("Existing job title not found in the document.")